In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import SVG
from matplotlib.patches import Rectangle

from vizopt.animation import SnapshotCallback, snapshots_to_animated_svg
from vizopt.base import OptimConfig
from vizopt.examples.sets import make_british_islands_graph
from vizopt.templates.euler.stars_vs_rectangles import EulerDiagramRect

In [ ]:
inclusion_graph = make_british_islands_graph(include_ireland_island=True)
inclusion_graph

In [ ]:
# Add hw/hh to leaf nodes — squares whose half-side equals the circle radius,
# keeping the same scale as the original circles-based layout.
G = inclusion_graph.copy()
for n in G.nodes:
    if G.out_degree(n) == 0:
        r = G.nodes[n]["r"]
        G.nodes[n]["hw"] = r
        G.nodes[n]["hh"] = r

In [ ]:
callback = SnapshotCallback(every=50)
cfg = OptimConfig(n_iters=3000, learning_rate=0.01)

optim = EulerDiagramRect.from_graph(
    G,
    weight_exclusion=20.0,
    weight_convexity=0.1,
    weight_position_anchor=0.01,
    weight_rect_collision=100.0,
    offsets=0.2,
)
optim.optimize(cfg, callback=callback)
history = optim.result_.history
problem = optim.problem_
named_results = {name: res for name, res in zip(optim.set_names, optim.sets_)}
rects_arr = optim.rects_
named_rects_out = {name: rects_arr[i] for i, name in enumerate(optim.leaf_names)}

In [ ]:
set_names = optim.set_names
leaf_names = optim.leaf_names

fig, ax = plt.subplots(figsize=(10, 10))

# Set boundaries — draw outermost (largest) first so inner ones render on top
for sname in reversed(set_names):
    res = named_results[sname]
    color = G.nodes[sname].get("color", "#888888")
    angles_k = res["angles"]
    cx, cy = res["center"]
    radii_k = res["radii"]
    xs = cx + radii_k * np.cos(angles_k)
    ys = cy + radii_k * np.sin(angles_k)
    ax.fill(np.append(xs, xs[0]), np.append(ys, ys[0]), alpha=0.12, color=color)
    ax.plot(np.append(xs, xs[0]), np.append(ys, ys[0]), color=color, lw=1.5, label=sname)

# Leaf rectangles
for lname in leaf_names:
    arr = named_rects_out[lname]
    cx, cy, hw, hh = arr
    color = G.nodes[lname].get("color", "#4472c4")
    ax.add_patch(Rectangle(
        [cx - hw, cy - hh], 2 * hw, 2 * hh,
        facecolor=color, edgecolor="white", alpha=0.85, linewidth=1,
    ))
    ax.text(cx, cy, lname.replace(" ", "\n"), ha="center", va="center",
            fontsize=6.5, color="white", fontweight="bold")

ax.set_aspect("equal")
ax.autoscale()
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.set_title("British Islands — Convex Star-Shaped Boundaries (Rectangle Elements)")
fig.tight_layout()
plt.show()

In [ ]:
pd.DataFrame(history).set_index("iteration").plot(figsize=(12, 4), title="Loss history")
plt.tight_layout()
plt.show()

In [ ]:
svg = snapshots_to_animated_svg(problem, callback.snapshots, size=600)
SVG(svg)

## Label rectangle optimization

Each set boundary reserves a floating label rectangle inside itself. The label position is an optimization variable — the optimizer pushes it toward the top of the boundary while keeping it inside the star and away from the leaf rectangles.

In [ ]:
callback_label = SnapshotCallback(every=50)
cfg_label = OptimConfig(n_iters=3000, learning_rate=0.01)

optim_label = EulerDiagramRect.from_graph(
    G,
    weight_exclusion=20.0,
    weight_convexity=0.1,
    weight_position_anchor=0.01,
    weight_smoothness=5.0,
    weight_rect_collision=100.0,
    label_rect_size=(0.4, 0.15),
    weight_label_enclosure=20.0,
    weight_label_element_exclusion=20.0,
    weight_label_collision=20.0,
    weight_label_top=2.0,
    rect_collision_alpha=0.5,
)
optim_label.optimize(cfg_label, callback=callback_label)
history_label = optim_label.result_.history
problem_label = optim_label.problem_
named_results_label = {name: res for name, res in zip(optim_label.set_names, optim_label.sets_)}
rects_label_arr = optim_label.rects_
named_rects_label = {name: rects_label_arr[i] for i, name in enumerate(optim_label.leaf_names)}

In [ ]:
label_hw, label_hh = 0.4, 0.15

fig, ax = plt.subplots(figsize=(10, 10))

for sname in reversed(set_names):
    res = named_results_label[sname]
    color = G.nodes[sname].get("color", "#888888")
    angles_k = res["angles"]
    cx, cy = res["center"]
    radii_k = res["radii"]
    xs = cx + radii_k * np.cos(angles_k)
    ys = cy + radii_k * np.sin(angles_k)
    ax.fill(np.append(xs, xs[0]), np.append(ys, ys[0]), alpha=0.12, color=color)
    ax.plot(np.append(xs, xs[0]), np.append(ys, ys[0]), color=color, lw=1.5)

    lx, ly = res["label_center"]
    ax.add_patch(Rectangle(
        [lx - label_hw, ly - label_hh], 2 * label_hw, 2 * label_hh,
        facecolor=color, edgecolor="white", alpha=0.85, linewidth=1, zorder=3,
    ))
    ax.text(lx, ly, sname, ha="center", va="center",
            fontsize=7.5, color="white", fontweight="bold", zorder=4)

for lname in leaf_names:
    arr = named_rects_label[lname]
    cx, cy, hw, hh = arr
    color = G.nodes[lname].get("color", "#4472c4")
    ax.add_patch(Rectangle(
        [cx - hw, cy - hh], 2 * hw, 2 * hh,
        facecolor=color, edgecolor="white", alpha=0.85, linewidth=1,
    ))
    ax.text(cx, cy, lname.replace(" ", "\n"), ha="center", va="center",
            fontsize=6.5, color="white", fontweight="bold")

ax.set_aspect("equal")
ax.autoscale()
ax.set_title("British Islands — Star-Shaped Boundaries with Label Rectangles")
fig.tight_layout()
plt.show()

In [ ]:
svg = snapshots_to_animated_svg(problem_label, callback_label.snapshots, size=600)
SVG(svg)